# 30문항으로 36개 검색 설정 평가하기

이 노트북은 공개 PDF 6건과 Ch11의 30문항을 사용합니다. 첫 셀부터 순서대로 실행합니다.


## 입력 준비

Ch11의 30문항과 정답 문서 ID를 직접 읽고 Ch12 배포 복제본과 같은지 확인합니다. 각 로그 행에는 설정·평균 지표·분할별 집계·질문별 결과를 함께 저장합니다.

### 설정 선택과 최종 확인을 분리하기

36개 설정은 개발 질문으로 비교하고 선택한 뒤 시험 질문을 한 번만 실행합니다. 마지막 통합 RAG는 PDF 읽기부터 생성까지 연결되는지 보는 별도 스모크 실행입니다. 검색기·질문 수·평가 단위가 다르므로 앞의 TF-IDF 설정과 수치 우열을 비교하지 않습니다.


In [1]:
from pathlib import Path
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "labs/part3/ch12-retrieval-tuning").exists())
data_path = root / "labs/part3/ch12-retrieval-tuning/data/advanced-documents.json"
query_path = root / "labs/part3/ch11-evaluation/data/evaluation-queries.jsonl"
copy_path = root / "labs/part3/ch12-retrieval-tuning/data/advanced-queries.json"
data = json.loads(data_path.read_text())
queries = [json.loads(line) for line in query_path.read_text().splitlines() if line]
distributed_queries = json.loads(copy_path.read_text())
assert queries == distributed_queries, "Ch11 질문 원본과 Ch12 배포 복제본이 다릅니다."
chunks, parents = data["chunks"], {item["doc_id"]: item for item in data["parents"]}
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)
matrix = vectorizer.fit_transform([item["text"] for item in chunks])
print(f"부모 문서 {len(parents)}건 · 자식 청크 {len(chunks)}개 · 평가 질문 {len(queries)}개")

부모 문서 6건 · 자식 청크 175개 · 평가 질문 30개


## 지표와 평가 함수

Recall@50은 후보 포함 여부, Precision@5와 nDCG@5는 상위 순서를 확인합니다.


In [2]:
def eligible_items(subset):
    eligible = [item for item in subset if item.get("answerable", True) and item["gold_doc_ids"]]
    if not eligible:
        raise ValueError("평가 가능한 문항이 없습니다")
    return eligible

def metrics(candidate_ids, top_ids, gold_ids):
    gold = set(gold_ids)
    recall = len(gold & set(candidate_ids[:50])) / len(gold) if gold else 0.0
    recall_5 = len(gold & set(top_ids[:5])) / len(gold) if gold else 0.0
    precision = sum(doc_id in gold for doc_id in top_ids[:5]) / 5
    gains = [1.0 if doc_id in gold else 0.0 for doc_id in top_ids[:5]]
    dcg = sum(gain / np.log2(rank + 2) for rank, gain in enumerate(gains))
    ideal = sum(1 / np.log2(rank + 2) for rank in range(min(len(gold), 5)))
    return recall_5, recall, precision, dcg / ideal if ideal else 0.0

## MMR 선택 함수

각 실험에서 같은 직접 구현을 사용합니다.


In [3]:
def mmr(query_vector, candidate_matrix, k=5, lambda_mult=0.5):
    relevance = cosine_similarity(query_vector, candidate_matrix)[0]
    selected, remaining = [], list(range(candidate_matrix.shape[0]))
    while remaining and len(selected) < k:
        if not selected:
            choice = max(remaining, key=lambda i: relevance[i])
        else:
            redundancy = cosine_similarity(candidate_matrix[remaining], candidate_matrix[selected]).max(axis=1)
            values = lambda_mult * relevance[remaining] - (1 - lambda_mult) * redundancy
            choice = remaining[int(np.argmax(values))]
        selected.append(choice)
        remaining.remove(choice)
    return selected

## 설정 한 개 평가

개발 문항과 시험 문항을 섞지 않도록 평가 대상을 인자로 받습니다.


In [4]:
def evaluate_config(config, subset):
    eligible = eligible_items(subset)
    vec = TfidfVectorizer(analyzer=config["analyzer"], ngram_range=config["ngram"]).fit([c["text"] for c in chunks])
    docs, values, results = vec.transform([c["text"] for c in chunks]), [], []
    for item in eligible:
        qv = vec.transform([item["question"]])
        base = cosine_similarity(qv, docs)[0]
        candidates = np.argsort(-base)[:50]
        chosen = mmr(qv, docs[candidates], k=config["report_k"], lambda_mult=config["lambda"])
        candidate_ids = list(dict.fromkeys(chunks[idx]["parent_id"] for idx in candidates))
        top_results, seen = [], set()
        for pos in chosen:
            chunk = chunks[candidates[pos]]
            if chunk["parent_id"] not in seen:
                top_results.append({"doc_id": chunk["parent_id"], "page": chunk["page"], "chunk_id": chunk["chunk_id"], "rank": len(top_results) + 1})
                seen.add(chunk["parent_id"])
        top_ids = [row["doc_id"] for row in top_results]
        scores = metrics(candidate_ids, top_ids, item["gold_doc_ids"])
        values.append(scores)
        results.append({"qid": item["qid"], "tier": item["tier"], "split": item["split"], "gold_doc_ids": item["gold_doc_ids"], "returned_count": len(top_results), "candidate_doc_ids": candidate_ids, "top_results": top_results, "recall_at_5": scores[0], "recall_at_50": scores[1], "precision_at_5": scores[2], "ndcg_at_5": scores[3]})
    mean = np.mean(values, axis=0)
    metric_names = ("recall_at_5", "recall_at_50", "precision_at_5", "ndcg_at_5")
    summary = {name: float(mean[i]) for i, name in enumerate(metric_names)}
    slices = {tier: {name: float(np.mean([row[name] for row in results if row["tier"] == tier])) for name in metric_names} for tier in sorted({row["tier"] for row in results})}
    return summary, slices, results

## 36개 개발 설정 실행

문자 분석기, n-gram, 보고 순위, MMR λ를 조합합니다. 후보 수는 50으로 고정합니다. 개발 문항으로 설정을 고른 뒤 시험 문항은 한 번만 평가합니다.


In [5]:
from datetime import date, datetime
from itertools import product
from hashlib import sha256
from time import perf_counter

fingerprint = sha256(data_path.read_bytes()).hexdigest()
query_fingerprint = sha256(query_path.read_bytes()).hexdigest()
session_id = datetime.now().strftime("%Y%m%dT%H%M%S")
rows = []
for analyzer, ngram, report_k, lam in product(("char", "char_wb"), ((2, 3), (2, 4), (3, 5)), (3, 5), (0.4, 0.8, 1.0)):
    settings = {"analyzer": analyzer, "ngram": ngram, "candidate_k": 50, "candidate_unit": "chunk", "report_k": report_k, "report_unit": "unique-parent-document", "lambda": lam}
    dev = [item for item in queries if item["split"] == "dev"]
    started = perf_counter()
    result, slices, details = evaluate_config(settings, dev)
    rows.append({"run_id": f"advanced-continuity-{session_id}-dev-{len(rows)+1:02d}", "run_date": str(date.today()), "source_chapter": "ch12", "query_source_chapter": "ch11", "query_source": str(query_path.relative_to(root)), "corpus_fingerprint": fingerprint, "query_fingerprint": query_fingerprint, "settings": settings, "evaluation_unit": "unique-parent-document", "metrics": result, "slices": slices, "results": details, "duration_ms": round((perf_counter() - started) * 1000, 2), "split": "dev", "query_count": len(dev), "excluded_query_count": len(dev) - len(details), "metrics_policy": "chunk-candidate-unique-parent-macro-v1", "comparison_policy": "compare-only-pairs-that-differ-by-one-setting"})
print("실험 로그 수:", len(rows))
print("첫 행 요약:", {key: rows[0][key] for key in ("run_id", "settings", "metrics", "query_count")})

실험 로그 수: 36
첫 행 요약: {'run_id': 'advanced-continuity-20260910T020805-dev-01', 'settings': {'analyzer': 'char', 'ngram': (2, 3), 'candidate_k': 50, 'candidate_unit': 'chunk', 'report_k': 3, 'report_unit': 'unique-parent-document', 'lambda': 0.4}, 'metrics': {'recall_at_5': 1.0, 'recall_at_50': 1.0, 'precision_at_5': 0.20000000000000004, 'ndcg_at_5': 0.9753953169047639}, 'query_count': 15}


## 로그 저장과 개발 설정 선택

원자료를 JSONL로 보존하고 개발 nDCG@5가 가장 높은 행을 확인합니다.


In [6]:
run_dir = root / "labs/part3/ch12-retrieval-tuning/runs"
run_dir.mkdir(parents=True, exist_ok=True)
session_dir = run_dir / "continuity" / session_id
session_dir.mkdir(parents=True, exist_ok=False)
path = run_dir / "advanced-continuity-experiment-log.jsonl"
experiment_payload = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows)
path.write_text(experiment_payload)
(session_dir / path.name).write_text(experiment_payload)
best = max(rows, key=lambda row: row["metrics"]["ndcg_at_5"])
test = [item for item in queries if item["split"] == "test"]
test_started = perf_counter()
test_metrics, test_slices, test_results = evaluate_config(best["settings"], test)
test_path = run_dir / "advanced-continuity-test-result.json"
test_report = {"run_id": f"advanced-continuity-{session_id}-test-01", "run_date": str(date.today()), "selected_run_id": best["run_id"], "source_chapter": "ch12", "query_source_chapter": "ch11", "query_source": str(query_path.relative_to(root)), "settings": best["settings"], "evaluation_unit": "unique-parent-document", "metrics": test_metrics, "slices": test_slices, "results": test_results, "duration_ms": round((perf_counter() - test_started) * 1000, 2), "split": "test", "query_count": len(test), "excluded_query_count": len(test) - len(test_results), "corpus_fingerprint": fingerprint, "query_fingerprint": query_fingerprint, "metrics_policy": "chunk-candidate-unique-parent-macro-v1", "comparison_policy": "selected-on-dev-then-evaluate-test-once"}
test_payload = json.dumps(test_report, ensure_ascii=False, indent=2)
test_path.write_text(test_payload)
(session_dir / test_path.name).write_text(test_payload)
print("저장:", path)
print("시험 결과 저장:", test_path)
print("개발 nDCG@5가 가장 높은 설정:", best["settings"])
print("개발 지표:", best["metrics"])
print("시험 지표:", test_metrics, "문항", len(test))
assert len(rows) >= 30 and len(dev) + len(test) == 30

저장: [사용자 폴더]/working/claude/projects/한국경제신문/labs/part3/ch12-retrieval-tuning/runs/advanced-continuity-experiment-log.jsonl
시험 결과 저장: [사용자 폴더]/working/claude/projects/한국경제신문/labs/part3/ch12-retrieval-tuning/runs/advanced-continuity-test-result.json
개발 nDCG@5가 가장 높은 설정: {'analyzer': 'char', 'ngram': (2, 3), 'candidate_k': 50, 'candidate_unit': 'chunk', 'report_k': 3, 'report_unit': 'unique-parent-document', 'lambda': 0.4}
개발 지표: {'recall_at_5': 1.0, 'recall_at_50': 1.0, 'precision_at_5': 0.20000000000000004, 'ndcg_at_5': 0.9753953169047639}
시험 지표: {'recall_at_5': 1.0, 'recall_at_50': 1.0, 'precision_at_5': 0.20000000000000004, 'ndcg_at_5': 0.9507906338095277} 문항 15


## 제공 PDF 6건 통합 실행

앞의 36개 행은 TF-IDF와 MMR 설정의 검색 평가입니다. 마지막은 설정군과 평가 단위가 다른 별도 스모크 실행으로, 개발용 Q002에서 PDF 쪽 본문을 검색·리랭킹·답변 생성에 실제로 전달합니다. 이 결과로 검색 지표나 생성 품질을 평가하지 않습니다.

통합 예제는 Ch10·Ch12 앞 임베딩 실습과 같은 `text-embedding-3-small`, 256차원을 사용하고, Ch6·Ch9와 같은 `gpt-5.4-mini`로 답변을 생성합니다. 차원은 실제 임베딩 배열로 확인해 통합 로그에 기록합니다.


In [7]:
from pypdf import PdfReader
from time import perf_counter

integration_started = perf_counter()
loaded_pages, integration_chunks = [], []
for parent in parents.values():
    pdf_path = root / "labs/data/rag-corpus" / parent["source"]
    pages = [page.extract_text() or "" for page in PdfReader(pdf_path).pages]
    for page_no, text in enumerate(pages, 1):
        loaded_pages.append(text)
        integration_chunks.append({"chunk_id": f"{parent['doc_id']}-P{page_no:02d}-C01", "parent_id": parent["doc_id"], "page": page_no, "text": text})
print("PDF 6건에서 읽은 페이지:", len(loaded_pages))
assert len(loaded_pages) >= 30 and all(page.strip() for page in loaded_pages)
print("통합 검색 입력 청크:", len(integration_chunks))

PDF 6건에서 읽은 페이지: 35
통합 검색 입력 청크: 35


## Dense 검색과 Kiwi BM25

OpenAI 임베딩과 한국어 형태소 기반 BM25를 PDF에서 직접 읽은 같은 35개 쪽 본문에서 각각 실행합니다.


In [8]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

load_dotenv(root / ".env")
target = next(item for item in queries if item["qid"] == "Q002")
assert target["split"] == "dev"
embedding = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=256)
dense_docs = np.array(embedding.embed_documents([item["text"] for item in integration_chunks]))
dense_query = np.array(embedding.embed_query(target["question"]))
dense_scores = cosine_similarity([dense_query], dense_docs)[0]
kiwi = Kiwi()
tokenized = [[token.form for token in kiwi.tokenize(item["text"])] for item in integration_chunks]
bm25 = BM25Okapi(tokenized)
sparse_scores = bm25.get_scores([token.form for token in kiwi.tokenize(target["question"])])
assert dense_docs.shape == (len(integration_chunks), 256)
print("dense·sparse 점수:", len(dense_scores), len(sparse_scores), "차원:", dense_docs.shape[1])

dense·sparse 점수: 35 35 차원: 256


## 직접 RRF와 CrossEncoder

두 순위의 상위 50개를 RRF로 합친 뒤 다국어 리랭커가 상위 10개만 다시 점수화합니다.


In [9]:
def rrf_pair(first, second, rank_constant=60):
    scores = {}
    for ranking in (first, second):
        for rank, idx in enumerate(ranking, 1):
            scores[idx] = scores.get(idx, 0.0) + 1 / (rank_constant + rank)
    return sorted(scores, key=scores.get, reverse=True)

dense_rank = list(np.argsort(-dense_scores)[:50])
sparse_rank = list(np.argsort(-sparse_scores)[:50])
rrf_candidates = rrf_pair(dense_rank, sparse_rank)[:10]
from sentence_transformers import CrossEncoder
ce = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=256, device="cpu")
ce_scores = ce.predict([(target["question"], integration_chunks[i]["text"]) for i in rrf_candidates], show_progress_bar=False)
reranked = [rrf_candidates[pos] for pos in np.argsort(-ce_scores)]
print("RRF→CE 상위 청크:", [integration_chunks[i]["chunk_id"] for i in reranked[:5]])

[사용자 폴더]/working/claude/projects/한국경제신문/labs/part3/ch12-retrieval-tuning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 8102.60it/s]

RRF→CE 상위 청크: ['E01-P03-C01', 'E01-P04-C01', 'E01-P02-C01', 'E02-P03-C01', 'E01-P05-C01']


## 검색된 쪽 본문과 출처를 답변에 전달

같은 문서를 한 번만 선택하고, 모델에는 doc_id·page·chunk_id와 그 출처의 쪽 본문을 함께 전달합니다.


In [10]:
retrieved, seen = [], set()
for idx in reranked:
    child = integration_chunks[idx]
    if child["parent_id"] in seen:
        continue
    retrieved.append({"doc_id": child["parent_id"], "page": child["page"], "chunk_id": child["chunk_id"], "text": child["text"]})
    seen.add(child["parent_id"])
    if len(retrieved) == 2:
        break
context = "\n\n".join(f"[{item['doc_id']} p.{item['page']} {item['chunk_id']}]\n{item['text']}" for item in retrieved)
print("부모 문서:", [(item["doc_id"], item["page"], item["chunk_id"]) for item in retrieved])

부모 문서: [('E01', 3, 'E01-P03-C01'), ('E02', 3, 'E02-P03-C01')]


In [11]:
from langchain.chat_models import init_chat_model

chat = init_chat_model("openai:gpt-5.4-mini", temperature=0)
prompt = f"""아래 자료만 사용해 질문에 한국어로 답하세요.
답 끝에 [doc_id p.쪽] 형식으로 출처를 표시하세요.

질문: {target['question']}

자료:
{context}"""
answer = chat.invoke(prompt).text.strip()
print("답변:", answer)

답변: 도서는 한 번에 **7권까지** 빌릴 수 있고, **14일 동안** 대출할 수 있습니다. [E01 p.3]


## 통합 결과 저장

질문, 검색한 출처, 실제 응답, 전체 시간을 한 JSON 파일로 보존합니다.


In [12]:
integrated = {
    "run_id": f"advanced-continuity-{session_id}-integrated-01",
    "run_type": "generation_smoke_test",
    "qid": target["qid"], "split": target["split"], "question": target["question"], "gold_doc_ids": target["gold_doc_ids"],
    "retrieved": [{k: item[k] for k in ("doc_id", "page", "chunk_id")} for item in retrieved],
    "response": answer,
    "duration_ms": round((perf_counter() - integration_started) * 1000, 2),
    "retrieval_metrics": None,
    "generation_evaluation": {"status": "not_evaluated", "next_chapter": "Ch15"},
    "settings": {"input": "PDF page text", "dense": "text-embedding-3-small", "sparse": "Kiwi+BM25", "fusion": "RRF(k=60)", "reranker": "BAAI/bge-reranker-v2-m3", "context_selection": "matched-page-text", "chat": "gpt-5.4-mini", "dimensions": dense_docs.shape[1]},
}
integrated_path = run_dir / "advanced-continuity-integrated-result.json"
integrated_payload = json.dumps(integrated, ensure_ascii=False, indent=2)
integrated_path.write_text(integrated_payload)
(session_dir / integrated_path.name).write_text(integrated_payload)
print("통합 결과:", integrated_path)
print("전체 시간:", integrated["duration_ms"], "ms")

통합 결과: [사용자 폴더]/working/claude/projects/한국경제신문/labs/part3/ch12-retrieval-tuning/runs/advanced-continuity-integrated-result.json
전체 시간: 19652.05 ms
